In [1]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
n = 40

df = pd.DataFrame({
    "Student_ID": range(1, n + 1),
    "Branch": np.random.choice(["CSE", "ECE", "ME", "CE"], n),
    "Study_Hours": np.round(np.random.uniform(1, 8, n), 1),
    "Attendance": np.random.randint(55, 100, n)
})

df["Marks"] = np.clip(
    np.round(
        30 + 5 * df["Study_Hours"]
        + 0.2 * df["Attendance"]
        + np.random.normal(0, 6, n)
    ),
    0, 100
).astype(int)

df.head()

,Student_ID,Branch,Study_Hours,Attendance,Marks
0,1,ME,5.3,70,75
1,2,CE,2.0,99,68
2,3,CSE,3.0,72,59
3,4,ME,3.6,78,70
4,5,ME,4.2,80,69


Task 1

In [2]:
def get_grade(marks):
    if marks >= 75:
        return "A"
    if marks >= 60:
        return "B"
    if marks >= 40:
        return "C"
    return "F"

df["Grade"] = df["Marks"].apply(get_grade)

print(df["Grade"].value_counts())

Grade
B    16
A    12
C    12
Name: count, dtype: int64


Task 2

In [3]:
low_attendance_marks = [
    marks
    for marks, attendance in zip(df["Marks"], df["Attendance"])
    if attendance < 75
]

marks_above_70 = list(
    filter(lambda marks: marks > 70, df["Marks"])
)

graced_marks = list(
    map(lambda marks: min(marks + 5, 100), df["Marks"])
)

print(low_attendance_marks)
print(marks_above_70)
print(graced_marks)

[75, 59, 60, 57, 78, 82, 55, 80, 48, 64, 54, 58, 65, 82, 80, 69]
[75, 74, 81, 72, 78, 82, 80, 87, 78, 78, 83, 82, 80, 84]
[80, 73, 64, 75, 74, 79, 69, 86, 77, 65, 66, 62, 59, 83, 87, 72, 60, 65, 85, 68, 53, 69, 59, 92, 64, 83, 63, 83, 73, 64, 88, 70, 87, 85, 74, 89, 52, 60, 55, 65]


Task 3

In [4]:
def summary(*values, round_to=2):
    if not values:
        return {
            "count": 0,
            "minimum": None,
            "maximum": None,
            "average": None
        }

    return {
        "count": len(values),
        "minimum": min(values),
        "maximum": max(values),
        "average": round(sum(values) / len(values), round_to)
    }

print(summary(10, 20, 30))
print(summary(*df["Marks"]))

{'count': 3, 'minimum': 10, 'maximum': 30, 'average': 20.0}
{'count': 40, 'minimum': 47, 'maximum': 87, 'average': 66.92}


Task 4

In [5]:
df.to_csv("students.csv", index=False)

def load_data(filename):
    try:
        return pd.read_csv(filename)
    except FileNotFoundError:
        print(f"File '{filename}' was not found.")
        return None

print(load_data("students.csv").head())
print(load_data("missing.csv"))

   Student_ID Branch  Study_Hours  Attendance  Marks Grade
0           1     ME          5.3          70     75     A
1           2     CE          2.0          99     68     B
2           3    CSE          3.0          72     59     C
3           4     ME          3.6          78     70     B
4           5     ME          4.2          80     69     B
File 'missing.csv' was not found.
None


Task 5

In [6]:
branch_averages = df.groupby("Branch")[["Marks", "Study_Hours"]].mean()

print(branch_averages)

print("Highest average marks:", branch_averages["Marks"].idxmax())

print(df.sort_values("Marks", ascending=False).head(5))

            Marks  Study_Hours
Branch                        
CE      68.416667       4.4500
CSE     67.200000       4.6600
ECE     65.625000       3.7875
ME      65.900000       4.2200
Highest average marks: CE
    Student_ID Branch  Study_Hours  Attendance  Marks Grade
23          24     CE          7.4          89     87     A
35          36    CSE          7.5          88     84     A
30          31    CSE          7.8          82     83     A
14          15     CE          7.8          63     82     A
32          33     CE          7.6          63     82     A


Task 6

In [7]:
marks = df["Marks"].to_numpy()

z_scores = (marks - marks.mean()) / marks.std()

print(z_scores)

print("Z-scores above 1:", np.sum(z_scores > 1))

[ 0.74346049  0.09897462 -0.72965008  0.28311344  0.19104403  0.65139108
 -0.26930302  1.29587695  0.46725226 -0.63758067 -0.54551126 -0.9137889
 -1.18999713  1.01966872  1.38794636  0.00690521 -1.09792772 -0.63758067
  1.20380754 -0.36137243 -1.74241359 -0.26930302 -1.18999713  1.84829341
 -0.72965008  1.01966872 -0.82171949  1.01966872  0.09897462 -0.72965008
  1.48001577 -0.17723361  1.38794636  1.20380754  0.19104403  1.57208518
 -1.834483   -1.09792772 -1.55827477 -0.63758067]
Z-scores above 1: 11


Task 7

In [8]:
q1 = df["Marks"].quantile(0.25)
q3 = df["Marks"].quantile(0.75)

iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

outliers = df[
    (df["Marks"] < lower_limit) |
    (df["Marks"] > upper_limit)
]

print("Mean:", df["Marks"].mean())
print("Median:", df["Marks"].median())
print("Mode:", df["Marks"].mode().tolist())
print("Variance:", df["Marks"].var())
print("Standard deviation:", df["Marks"].std())
print("Minimum:", df["Marks"].min())
print("Maximum:", df["Marks"].max())

print("Q1:", q1, "Q3:", q3, "IQR:", iqr)
print("Lower limit:", lower_limit, "Upper limit:", upper_limit)
print("Outliers:", outliers["Marks"].tolist())

Mean: 66.925
Median: 66.0
Mode: [59, 60, 78]
Variance: 120.99423076923075
Standard deviation: 10.99973775911184
Minimum: 47
Maximum: 87
Q1: 59.0 Q3: 78.0 IQR: 19.0
Lower limit: 30.5 Upper limit: 106.5
Outliers: []


Task 8

In [9]:
study_corr = df["Marks"].corr(df["Study_Hours"])
attendance_corr = df["Marks"].corr(df["Attendance"])

print("Study hours correlation:", study_corr)
print("Attendance correlation:", attendance_corr)

print(
    "Stronger relationship:",
    "Study_Hours" if abs(study_corr) > abs(attendance_corr)
    else "Attendance"
)

Study hours correlation: 0.8537360674373369
Attendance correlation: 0.06674385703072847
Stronger relationship: Study_Hours


Task 9

In [10]:
mean_marks = df["Marks"].mean()

sem = stats.sem(df["Marks"])

ci = stats.t.interval(
    0.95,
    df=len(df) - 1,
    loc=mean_marks,
    scale=sem
)

print("95% confidence interval:", ci)

95% confidence interval: (np.float64(63.407113198283106), np.float64(70.4428868017169))


Task 10

In [11]:
t_stat, p_value = stats.ttest_1samp(df["Marks"], 60)

print("t-statistic:", t_stat)
print("p-value:", p_value)

print(
    "Conclusion:",
    "Reject H0; the mean differs significantly from 60."
    if p_value < 0.05
    else "Fail to reject H0; insufficient evidence that the mean differs from 60."
)

t-statistic: 3.981689977749835
p-value: 0.00028925643020879
Conclusion: Reject H0; the mean differs significantly from 60.


Task 11

In [12]:
cse_marks = df.loc[df["Branch"] == "CSE", "Marks"]
ece_marks = df.loc[df["Branch"] == "ECE", "Marks"]

t_stat_2, p_value_2 = stats.ttest_ind(cse_marks, ece_marks)

print("CSE mean:", cse_marks.mean(), "ECE mean:", ece_marks.mean())
print("t-statistic:", t_stat_2)
print("p-value:", p_value_2)

print(
    "Conclusion:",
    "Reject H0; the groups differ significantly."
    if p_value_2 < 0.05
    else "Fail to reject H0; no statistically significant difference detected."
)

CSE mean: 67.2 ECE mean: 65.625
t-statistic: 0.2757747137287823
p-value: 0.7862511358871336
Conclusion: Fail to reject H0; no statistically significant difference detected.


Task 12

In [13]:
print(f"The class average is {mean_marks:.2f} marks, with a median of {df['Marks'].median():.2f}.")

print(
    f"The standard deviation is {df['Marks'].std():.2f}, "
    "showing the spread in performance."
)

print(
    f"The branch with the highest average is "
    f"{branch_averages['Marks'].idxmax()}."
)

print(
    f"Study hours correlate with marks at {study_corr:.3f}; "
    f"attendance correlates at {attendance_corr:.3f}."
)

print(
    f"{'Study hours' if abs(study_corr) > abs(attendance_corr) else 'Attendance'} "
    "has the stronger observed linear relationship with marks."
)

The class average is 66.92 marks, with a median of 66.00.
The standard deviation is 11.00, showing the spread in performance.
The branch with the highest average is CE.
Study hours correlate with marks at 0.854; attendance correlates at 0.067.
Study hours has the stronger observed linear relationship with marks.
